In [2]:
import pandas as pd

df = pd.read_csv("../data/raw/WA_Fn-UseC_-Telco-Customer-Churn.csv")

# 1. TotalCharges: text -> number. Blanks are new customers (tenure 0), not yet billed.

In [3]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"].str.strip(), errors="coerce")
assert (df.loc[df["TotalCharges"].isna(), "tenure"] == 0).all()
df["TotalCharges"] = df["TotalCharges"].fillna(0)

# 2. Inconsistent categories: "No internet service" / "No phone service" are just "No"
#    (already captured by InternetService / PhoneService)

In [4]:
service_cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup", "DeviceProtection",
                "TechSupport", "StreamingTV", "StreamingMovies"]
df[service_cols] = df[service_cols].replace(
    {"No internet service": "No", "No phone service": "No"}
)

# 3. Target: Yes/No -> 1/0

In [5]:
df["Churn"] = df["Churn"].map({"No": 0, "Yes": 1})

# 4. Duplicates check, then drop the identifier (no predictive meaning)

In [6]:
print("Duplicate rows:", df.duplicated().sum())
df = df.drop(columns="customerID")

Duplicate rows: 0


# 5. Verify

In [7]:
print(df.shape)
print(df.isna().sum().sum(), "missing values")
print(df.dtypes.value_counts())
print(df["Churn"].mean().round(3))
for c in service_cols:
    print(c, df[c].unique())

(7043, 20)
0 missing values
object     15
int64       3
float64     2
Name: count, dtype: int64
0.265
MultipleLines ['No' 'Yes']
OnlineSecurity ['No' 'Yes']
OnlineBackup ['Yes' 'No']
DeviceProtection ['No' 'Yes']
TechSupport ['No' 'Yes']
StreamingTV ['No' 'Yes']
StreamingMovies ['No' 'Yes']


# 6. Save cleaned data

In [8]:
df.to_csv("../data/processed/telco_clean.csv", index=False)